In [19]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


In [21]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [23]:
len(splitted_docs)

26

In [24]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)


### agent - tools,llm,prompt

In [34]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    docs = vector_store.similarity_search(query=query,k=4)
    context = "" 
    for doc in docs:
        context += doc.page_content + "\n\n" 
    return context

In [35]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [36]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [37]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [39]:
query = "what is the doctor name and patient name?"
response = agent.invoke({"messages":[{"role":"user","content":query}]})
print(response["messages"][-1].content)

The doctor's name is Dr. Kiran Bhargava Pathak, and the patient's name is Ms. Nikita Chudhary.
